#### DFS/Backtrack + DP/Memoization

- 時間複雜度：$O(D \cdot L + n^3 + 2^n \cdot n)$
  - 預處理階段：$O(D \cdot L)$，其中 $D$ 是字典單字量，$L$ 是單字平均長度。我們需遍歷每個單字進行摩斯碼轉換。
  - 遞迴與切割：
    - 子問題數量為 $n$（字串長度）。
    - 每個子問題內有一個 $O(n)$ 的迴圈與 $O(n)$ 的字串切片（target_code），基本結構開銷為 $O(n^3)$。
  - 組合生成：最糟情況，如果字典包含所有可能的子字串（例如 s = "aaaa", dict = ["a", "aa", "aaa", "aaaa"]），組合的數量會呈現指數級 $O(2^n)$ 增長，且每個組合理由$O(n)$個字組成。(參考140題)
- 空間複雜度：$O(D \cdot L + n + 2^n \cdot n)$
  - 字典雜湊表：$O(D \cdot L)$，儲存所有轉換後的摩斯碼及其對應單字。
  - 遞迴堆疊：最深為 $O(n)$。
  - 記憶化快取 (Memo)：這是最大的消耗。在最差情況下，我們需要儲存 $2^{n-1}$ 種可能的路徑組合，且每個路徑包含的單字總字元數約為 $O(n)$。

In [1]:
def translate_morse(morse_str, dictionary):
    print(f"{morse_str = }\n")

    # 1. 標準摩斯密碼映射表
    MORSE_MAP = {
        'A': '.-', 'B': '-...', 'C': '-.-.', 'D': '-..', 'E': '.', 'F': '..-.',
        'G': '--.', 'H': '....', 'I': '..', 'J': '.---', 'K': '-.-', 'L': '.-..',
        'M': '--', 'N': '-.', 'O': '---', 'P': '.--.', 'Q': '--.-', 'R': '.-.',
        'S': '...', 'T': '-', 'U': '..-', 'V': '...-', 'W': '.--', 'X': '-..-',
        'Y': '-.--', 'Z': '--..'
    }
    print(f"{MORSE_MAP = }\n")

    # 2. 預處理字典：將單字轉為摩斯碼，並處理「多個單字對應同一個摩斯碼」的情況
    # 格式: { "摩斯碼": ["單字1", "單字2"] }
    # Time: O(D * L), D 為字典單字數, L 為單字平均長度
    # Space: O(D * L), 儲存摩斯碼與單字的對應關係
    morse_to_words = {}
    for word in dictionary:
        word = word.upper()
        m_code = "".join([MORSE_MAP.get(char, "") for char in word])
        if m_code not in morse_to_words:
            morse_to_words[m_code] = []
        morse_to_words[m_code].append(word)
    print(f"{morse_to_words = }\n")

    # 3. 記憶化字典：紀錄 index -> 該位置起的所有可能翻譯組合
    # Space: O(2^n * n), 最差情況會有指數成長的組合，每個組合有 n 個 morse_str 長度
    # sub_solutions 是空的，就不會有組合。不是每個 dfs 都包含 O(2^n) 操作，而是 O(n^3) 基礎開銷 加上 額外開銷 O(2^n)
    memory = {}

    # time: O(n^3), space: O(n)
    # time: 子問題數量：O(n) (共有 n 個 start)
    # time: 單個子問題內的運算：迴圈 O(n) * 切片 O(n) = O(n^2)
    # time: 不含結果拼接的總開銷：O(n * n^2) = O(n^3)
    def backtrack(start): # time: O(n), space: O(n)
        print("-" * 100)
        print(f"{start = }")
        # 如果已經計算過此位置，直接回傳結果
        if start in memory:
            print(f"return memo[{start}]")
            return memory[start]
        
        # 基礎情況：如果到達字串末尾，回傳包含空列表的列表，表示一條路徑完成
        if start == len(morse_str):
            print(f"return [[]]")
            return [[]]

        result = []
        # 4. 嘗試從當前位置切出不同長度的摩斯碼
        for end in range(start + 1, len(morse_str) + 1): # time: O(n)
            target_code = morse_str[start:end] # time: O(n)
            
            # 如果這段代碼在我們的字典中
            if target_code in morse_to_words:
                # 取得該代碼對應的所有可能單字（處理歧義性）
                possible_words = morse_to_words[target_code]

                print(f"morse_str[{start}:{end}] = {target_code = }")
                print(f"{possible_words = }")
                
                # 遞迴獲取剩餘部分的所有組合
                sub_result = backtrack(end)
                print(f"{start = }, {sub_result = }")
                
                # 組合當前單字與後續解法
                for word in possible_words:
                    for sub in sub_result:
                        result.append([word] + sub)
                        print(f"add {[word]} + {sub} in {result}")
                
                print("_" * 100)
        
        memory[start] = result
        print(f"{result = }\n{memory = }")

        print("=" * 100)
        return result

    # 5. 執行並格式化結果
    all_paths = backtrack(0)
    all_path_str = [" ".join(path_list) for path_list in all_paths]
    return all_path_str

In [2]:
morse_input = "....---"
word_dict = ["H", "EE", "E", "S", "O", "SON", "SO"]
translate_morse(morse_input, word_dict)

morse_str = '....---'

MORSE_MAP = {'A': '.-', 'B': '-...', 'C': '-.-.', 'D': '-..', 'E': '.', 'F': '..-.', 'G': '--.', 'H': '....', 'I': '..', 'J': '.---', 'K': '-.-', 'L': '.-..', 'M': '--', 'N': '-.', 'O': '---', 'P': '.--.', 'Q': '--.-', 'R': '.-.', 'S': '...', 'T': '-', 'U': '..-', 'V': '...-', 'W': '.--', 'X': '-..-', 'Y': '-.--', 'Z': '--..'}

morse_to_words = {'....': ['H'], '..': ['EE'], '.': ['E'], '...': ['S'], '---': ['O'], '...----.': ['SON'], '...---': ['SO']}

----------------------------------------------------------------------------------------------------
start = 0
morse_str[0:1] = target_code = '.'
possible_words = ['E']
----------------------------------------------------------------------------------------------------
start = 1
morse_str[1:2] = target_code = '.'
possible_words = ['E']
----------------------------------------------------------------------------------------------------
start = 2
morse_str[2:3] = target_code = '.'
possible_words = ['E']
-------------

['E E E E O',
 'E E EE O',
 'E EE E O',
 'E S O',
 'E SO',
 'EE E E O',
 'EE EE O',
 'S E O',
 'H O']